In [ ]:
# Uninstall potentially conflicting pre-installed versions (optional, but ensures a clean state)
!pip uninstall -qqy protobuf six wheel typing-extensions

# Install a compatible version of protobuf
# Version 4.23.4 or 3.20.x are often good choices depending on the TF version used in Kaggle
!pip install protobuf==4.23.4


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Model
from tensorflow.keras.applications.efficientnet_v2 import EfficientNetV2S
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input
import numpy as np
from sklearn.metrics import classification_report, accuracy_score
from tensorflow.keras.utils import load_img, img_to_array

In [ ]:
IMG_SIZE = (224, 224)  # A good standard size
BATCH_SIZE = 32
SEED = 42
NUM_CLASSES = 10
train_dir = "/kaggle/input/new-paddy-doctor-paddy-disease-classification/paddy-disease-classification/train_images"
test_dir = "/kaggle/input/new-paddy-doctor-paddy-disease-classification/paddy-disease-classification/test_images"

In [ ]:
train_ds = keras.utils.image_dataset_from_directory(
    train_dir,
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = keras.utils.image_dataset_from_directory(
    test_dir,
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)
class_names = train_ds.class_names

In [ ]:
data_augmentation = keras.Sequential(
  [
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
    layers.RandomTranslation(height_factor=0.2, width_factor=0.2),
    # --- ADD THESE LINES ---
    layers.RandomContrast(0.1),  # Randomly adjust contrast by up to 10%
    layers.RandomBrightness(0.1), # Randomly adjust brightness by up to 10%
    # -----------------------
  ],
  name="data_augmentation"
)

In [ ]:
# Apply augmentation to the training dataset
train_ds = train_ds.map(lambda x, y: (data_augmentation(x, training=True), y),
                        num_parallel_calls=tf.data.AUTOTUNE)

# Apply the EfficientNet-specific preprocessing to BOTH datasets
train_ds = train_ds.map(lambda x, y: (preprocess_input(x), y),
                        num_parallel_calls=tf.data.AUTOTUNE)
val_ds = val_ds.map(lambda x, y: (preprocess_input(x), y),
                      num_parallel_calls=tf.data.AUTOTUNE)

# Configure datasets for performance
train_ds = train_ds.prefetch(buffer_size=tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=tf.data.AUTOTUNE)

In [ ]:
base_model = EfficientNetV2S(
    include_top=False, 
    weights='imagenet',   
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)
)

base_model.trainable = False


inputs = keras.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3))

x = base_model(inputs, training=False)

x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.5)(x) 
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x) # Final 10-class output


model_transfer_learning = Model(inputs, outputs)


model_transfer_learning.summary()

In [ ]:
lr_scheduler = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=1,
    min_lr=0.000001
)


In [ ]:
model_transfer_learning.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy', # Use this because your labels are integers
    metrics=['accuracy']
)

In [ ]:
history = model_transfer_learning.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=[lr_scheduler]
)

In [ ]:
# NEW CELL: Phase 2 - Fine-Tuning Setup

# 1. Unfreeze the base model
base_model.trainable = True

# 2. Freeze all but the last 30 layers 
# (Layers 0 to N-31 are frozen; the top 30 are trainable)
for layer in base_model.layers[:-80]:
    layer.trainable = False

# 3. Recompile the model with a very low learning rate
print("--- Recompiling model for Fine-Tuning ---")
model_transfer_learning.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001), # CRITICAL: Very low LR
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# 4. Continue training for another 15 epochs 
print("--- Starting Phase 2: Fine-Tuning (Unfrozen Top Layers) ---")
history_2 = model_transfer_learning.fit(
    train_ds,
    validation_data=val_ds,
    epochs=70, # Total target epochs (15 initial + 15 fine-tune)
    initial_epoch=history.epoch[-1] + 1, # <--- CORRECTED: Changed history_1 to history
    callbacks=[lr_scheduler]
)

In [ ]:
## NEW CELL: Final Evaluation

print("--- Running final evaluation on validation set ---")
# Evaluate on the validation dataset (val_ds)
loss, acc = model_transfer_learning.evaluate(val_ds, verbose=1)

print(f"\nFinal Validation Loss: {loss:.4f}")
print(f"Final Validation Accuracy: {acc*100:.2f}%")

In [ ]:
import numpy as np
from sklearn.metrics import classification_report
import tensorflow as tf

# 1. Collect all images and labels into sequential NumPy arrays
X_test = []
Y_true = []

# Iterate over a clean, unbatched version of the validation dataset to ensure order
for image, label in val_ds.unbatch().as_numpy_iterator():
    X_test.append(image)0
    Y_true.append(label)

X_test = np.array(X_test)
Y_true = np.array(Y_true)

print(f"Collected {len(X_test)} total validation images and labels.")

# 2. Make predictions on the collected NumPy array
# This ensures the predictions match the order of Y_true
print("--- Generating Predictions on Reconstructed Validation Data ---")
Y_pred_probabilities = model_transfer_learning.predict(X_test)
Y_pred_classes = np.argmax(Y_pred_probabilities, axis=1)

# 3. Print the final, corrected classification report
print("\n--- Corrected Detailed Classification Report (Final Model) ---")
print(classification_report(Y_true, Y_pred_classes, target_names=class_names))

# 4. Final Sanity Check
accuracy_check = np.mean(Y_pred_classes == Y_true) * 100
print(f"\nOverall Accuracy Check (from Report Data): {accuracy_check:.2f}%")

In [ ]:
# Cell 11: Modified Plotting Code

import matplotlib.pyplot as plt

# Combine metrics from both phases
acc = history_1.history['accuracy'] + history_2.history['accuracy']
val_acc = history_1.history['val_accuracy'] + history_2.history['val_accuracy']
loss = history_1.history['loss'] + history_2.history['loss']
val_loss = history_1.history['val_loss'] + history_2.history['val_loss']

# Plot Accuracy
plt.figure(figsize=(12, 5))
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.title('Combined Model Accuracy (Feature Extraction + Fine-Tuning)')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.axvline(x=15, color='r', linestyle='--', label='Fine-Tuning Starts') # Mark the transition
plt.legend()
plt.show()


# Plot Loss
plt.figure(figsize=(12, 5))
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.title('Combined Model Loss (Feature Extraction + Fine-Tuning)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.axvline(x=15, color='r', linestyle='--', label='Fine-Tuning Starts')
plt.legend()
plt.show()

In [ ]:
import tensorflow as tf
from tensorflow import keras

model_transfer_learning.save('my_trained_model.keras')
print("Model successfully saved to 'my_trained_model.keras'")
model_transfer_learning.save('my_trained_model.h5')
print("Model successfully saved to 'my_trained_model.h5'")

In [ ]:
from tensorflow.keras.models import load_model

# Load the saved model
loaded_model = load_model('my_trained_model.keras') # Or 'my_trained_model.h5'

# Verify the model architecture (optional)
loaded_model.summary()

# The model is now ready to be used!


In [ ]:
import numpy as np

np.save("class_names.npy", class_names)

print("Model and class names saved!")

In [ ]:
!pip install gradio

import gradio as gr
import tensorflow as tf
import numpy as np
from PIL import Image

from tensorflow.keras.applications.efficientnet_v2 import preprocess_input
 

model = tf.keras.models.load_model("my_paddy_model.keras")

class_names = np.load("class_names.npy", allow_pickle=True)

def predict_image(pil_image):
   
    img = pil_image.resize((224, 224))
    
    
    img_array = tf.keras.preprocessing.image.img_to_array(img)
    
    img_batch = np.expand_dims(img_array, axis=0)
    
    processed_image = preprocess_input(img_batch)
    
    
    prediction = model.predict(processed_image)
    
    scores = prediction[0]
    
    confidences = {class_names[i]: float(scores[i]) for i in range(len(class_names))}
    
    return confidences

iface = gr.Interface(
    fn=predict_image,
    inputs=gr.Image(type="pil", label="Upload Paddy Leaf Image"),
    outputs=gr.Label(num_top_classes=3, label="Top Predictions"),
    title="Paddy Disease Classifier",
    description="Upload an image of a paddy leaf to classify the disease. This model uses an EfficientNetV2S base."
)

iface.launch(share=True)